# Lesson 10 - យន្តការសិប្បនិម្មិតក្នុងការផលិត

នៅក្នុងមេរៀននេះ អ្នកនឹងរៀនអំពី **លំនាំផលិតកម្ម** សម្រាប់យន្តការសិប្បនិម្មិតដោយប្រើ Microsoft Agent Framework (Python)។ យើងគ្របដណ្តប់៖

- **ការសង្កេត** — ការបន្ថែមពេលវេលា និងកំណត់ហេតុទៅលើការបំពេញបេសកកម្មរបស់យន្តការ
- **ការវាយតម្លៃ** — ប្រើយន្តការវាយតម្លៃដើម្បីផ្តល់ពិន្ទុគុណភាពចម្លើយ
- **ការគ្រប់គ្រងថ្លៃដើម** — វិធីសាស្ត្រសម្រាប់ការបង្កើតអត្រាតូកិន និងជ្រើសរើសម៉ូដែល

សេណារីយោគឺជាយន្តការធ្វើដំណើរមួយដែលជួយអ្នកប្រើរៀបចំដំណើរធ្វើដំណើរ ដែលមានការត្រួតពិនិត្យ និងវាយតម្លៃជាភាគរយនៅលើ។


## ការដំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ការពិចារណានៃការផលិត

ការផ្លាស់ប្តូរអ្នកទំនាក់ទំនង AI ពីគំរូទៅផលិតកម្ម ត្រូវការយកចិត្តទុកដាក់យ៉ាងម៉ត់ចត់លើបីជើងសម្រុង៖

1. **ការសង្កេតយល់** — អ្នកត្រូវការមើលឃើញពីអ្វីដែលភ្នាក់ងារធ្វើ, រយៈពេលវេលា, និងឧបករណ៍ណាដែលវាហៅ។ ដោយគ្មានការតាមដាននិងកំណត់ហេតុ ការស្វែងរកកំហុសនៅក្នុងផលិតកម្មជ្រាបលំបាកខ្លាំង។

2. **ការវាយតម្លៃ** — ការត្រួតពិនិត្យគុណភាពដោយស្វ័យប្រវត្តិធានាថាចម្លើយរបស់ភ្នាក់ងារនៅតែត្រឹមត្រូវ, ពេញលេញ, និងមានប្រយោជន៍ក្នុងរយៈពេលវែង។ ភ្នាក់ងារវាយតម្លៃអាចពិន្ទុចម្លើយតាមលក្ខខណ្ឌដែលបានកំណត់។

3. **ការគ្រប់គ្រងថ្លៃដើម** — ការប្រើប្រាស់តូកិនមានឥទ្ធិពលโดยផ្ទាល់លើថ្លៃដើម។យុទ្ធសាស្រ្តដូចជា ការបង្កើនប្រសិទ្ធភាពបម្លាស់ប្តូរ, ជម្រើសម៉ូដែល, និងការផ្ទុកទិន្នន័យជាមុនជួយរក្សារចំណាយក្រោមការគ្រប់គ្រងដោយមិនបាត់បង់គុណភាព។


## ការស្ថាបនាតំណាងអាចមើលឃើញបាន

យើងកំណត់ឧបករណ៍ធ្វើដំណើរ ហើយបញ្ជូលការហៅតំណាងជាមួយពេលវេលាដើម្បីអាចតាមដានការពន្យារពេលបាន។ នៅក្នុងការផលិត អ្នកនឹងរួមបញ្ចូលជាមួយ OpenTelemetry ឬបណ្តាញតាមដានដូចគ្នា។


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## សំណុំបែបបទវាយតម្លៃ

រូបមន្តផលិតកម្មទូទៅគឺប្រើភ្នាក់ងារទីពីរជា **អ្នកវាយតម្លៃ**។ អ្នកវាយតម្លៃវាយតម្លៃចម្លើយរបស់ភ្នាក់ងារចម្បងប្រឆាំងនឹងលក្ខខណ្ឌដែលបានកំណត់ជាមុន ដូចជាការបញ្ចប់, ភាពត្រឹមត្រូវ, និងភាពជួយស្រួល។

នេះនាំឱ្យមាន៖
- ទ្វារគុណភាពជាអូតូម៉ាទិចមុនការឆ្លើយតបទៅដល់អ្នកប្រើ
- ការរកឃើញការវិវត្តវិញពេលបង្ហាញឬម៉ូដែលផ្លាស់ប្ដូរ
- ការត្រួតពិនិត្យជាបន្តបន្ទាប់លើប្រសិទ្ធភាពភ្នាក់ងារេពេលវេលា


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## វិធីសាស្ត្រគ្រប់គ្រងចំណាយ

ការគ្រប់គ្រងចំណាយមានសារៈសំខាន់សម្រាប់អ្នកភ្នាក់ងារចង់ផលិត AI។ នេះជា វិធីសាស្ត្រចម្បងៗ៖

| វិធីសាស្ត្រ | ការពិពណ៌នា |
|---|---|
| **ការប្រសើរឡើងនៃការផ្តល់បញ្ចូល** | រក្សាការណែនាំប្រព័ន្ធឲ្យខ្លី ។ លុបបំបាត់បរិបទដែលធ្វើកំហុសដើម្បីកាត់បន្ថយបញ្ចូល token ។ |
| **ជម្រើសម៉ូដែល** | ប្រើម៉ូដែលតូចៗ និងថោកជាង (ឧ. GPT-4o-mini) សម្រាប់ភារកិច្ចសាមញ្ញដូចជាការបែងចែកឬទាក់ទាញ និងរក្សាទុកម៉ូដែលធំសម្រាប់ការគិតលំបាក។ |
| **ការផ្ទុកទិន្នន័យស្តុក** | ផ្ទុកលទ្ធផលឧបករណ៍ និងសំណួរកម្មវិធីដែលប្រើម៉ោងញឹកញាប់ ដើម្បីចៀសវាងការហៅ API ជាបន្តបន្ទាប់។ |
| **ថវិកា token** | កំណត់ `max_tokens` ដើម្បីទប់ស្កាត់ចម្លើយប្រយ័ត្នច្រើនពេកដែលមិនរំពឹងទុក។ |
| **កំណត់ក្រុមបញ្ចូល** | ក្រុមសំណួរជាច្រើនរបស់អ្នកប្រើទៅក្នុងការហៅ API មួយ ប្រសិនបើអាចធ្វើបាន។ |

ក្នុងការអនុវត្ត វិធីសាស្ត្រចំណាត់ថ្នាក់គ្នារបស់វាដំណើរការល្អ ៖ ផ្ញើសំណើសាមញ្ញទៅម៉ូដែលមានល្បិចល្បាញ និងថោក ហើយបង្កើនទៅសំណួរលំបាកទៅម៉ូដែលដែលមានសមត្ថភាពខ្ពស់។


## សេចក្ដីសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀប៖

1. **បន្ថែមការសង្កេតតាម** ទៅលើប្រតិបត្ដិការប្រតិជ្ញាករ ជាមួយពេលវេលា និងការចុះបញ្ជី ដើម្បីដាក់មូលដ្ឋានសម្រាប់ការតាមដាន និងត្រួតពិនិត្យ។
2. **វាយតម្លៃការឆ្លើយតបរបស់ប្រតិជ្ញាករ** ជាស្វ័យប្រវត្តិ ដោយប្រើប្រតិជ្ញាករវាយតម្លៃដែលវាយតំលៃភាពពេញលេញ គុណភាព និងភាពមានប្រយោជន៍។
3. **គ្រប់គ្រងការចំណាយ** តាមរយៈការបង្កើតពាក្យបញ្ជារសម្រួល ការជ្រើសម៉ូដែល ការផ្ទុកជាមុន និងថវិកាអត្ថបទ។

គំរូផលិតកម្មទាំងនេះជួយធានាប្រតិជ្ញាកររបស់អ្នកឱ្យមានភាពទុកចិត្ត មានការវាស់វែងល្អ និងមានប្រសិទ្ធភាពតម្លៃនៅមូលដ្ឋានធំ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
